# SKU Percentage Finder

## Importing Library

In [354]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

## Import Necessary File

In [ ]:
JNP_File=r"folder\file.csv"
PDM_File=r"folder\file.csv"
AutocareAttrFile=r"folder\file.csv"


## Dataframe Creation

In [356]:
df_s=pd.read_csv(PDM_File)

In [ ]:
df_s

### Custom Cells (for PDM)

In [ ]:
print(df_s.columns)
df_s['AttributeUomLabel']=df_s['AttributeUomLabel'].str.lower()
df_s["Attribute_Full"] = np.where(df_s["AttributeUomLabel"].notna(), df_s['Attribute'] + " ("+df_s['AttributeUomLabel']+")", df_s["Attribute"])
df_s=df_s.astype(str)

df_s.rename(columns={
'PartType': 'PartName',
'AttributeValue': 'Value',
},inplace=True)



In [359]:
df_s['Key']=df_s['PartNumber']+df_s['PartName']+df_s['Attribute_Full']

In [ ]:
df_s

In [ ]:
df_Brands=df_s[['PartNumber','BrandName']].drop_duplicates(ignore_index=True)
df_Brands

In [ ]:
df_BrandMap=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Autocare\Active_Part_List.xlsx",sheet_name="BrandMap")
df_BrandMap=df_BrandMap[["BrandName","Brand"]]
df_BrandMap.dropna(inplace=True)
df_BrandMap

In [363]:
df_PNS=df_s[['PartNumber','PartName','BrandName']].drop_duplicates(ignore_index=True)

## AutocareList

In [364]:
df_AC=pd.read_csv(AutocareAttrFile)

In [ ]:
df_AC

In [366]:
df_AC=df_AC[[ 'PartTerminologyName','PAName', 'UOMLabel']]

In [367]:
df_AC.rename(columns={'PartTerminologyName': 'PartName','PAName': 'Attribute'},inplace=True)

In [368]:
#df_AC.dropna(subset=['Attribute']).reset_index(drop=True)

In [369]:
df_AC["Attribute_Full"] = np.where(df_AC["UOMLabel"].notna(), df_AC['Attribute'] + " ("+df_AC['UOMLabel']+")", df_AC["Attribute"])

In [370]:
df_AC=df_AC.drop_duplicates(ignore_index=True)

In [371]:
df_AC['Attribute_Type']="Autocare_Attribute"

In [ ]:
df_AC

## Merging

In [373]:
df_merged=df_PNS.merge(df_AC,how='left')
df_merged.shape

(5954360, 7)

In [ ]:
df_merged

In [375]:
df_merged['Key']=df_merged['PartNumber']+df_merged['PartName']+df_merged['Attribute_Full']

In [ ]:
df_merged

In [377]:
df_Final=df_merged.merge(df_s,how="outer")
df_Final.shape

(9798620, 10)

In [ ]:
df_Final

In [379]:
df_Final['Attributes']=np.where(df_Final["Attribute_Full"].notna(), df_Final["Attribute_Full"] , df_Final["Attribute"])
df_Final=df_Final[['PartNumber', 'BrandName','PartName', 'Attributes', 'Attribute_Type', 'Value']]
df_Final['AC_AttributeCount']=np.where(df_Final["Attribute_Type"].notna(), 1 , 0)
df_Final['Attribute_Value']=np.where((df_Final["Attribute_Type"].notna())  & (df_Final["Value"].notna()), 1 , 0)
# df_Final=df_Final.dropna(subset='Autocare')

df_Final['Attribute_Type']=np.where((df_Final["Attribute_Type"].notna())  , df_Final["Attribute_Type"] , 'FBG_Attribute')

In [ ]:
df_BrandMap

In [381]:
df_Final=df_Final.merge(df_BrandMap,how="inner")
df_Final.shape

(2737343, 9)

In [ ]:
df_Final

In [383]:
df_Final=df_Final[['PartNumber', 'PartName', 'Attributes', 'Attribute_Type',
       'Value', 'AC_AttributeCount', 'Attribute_Value', 'Brand']] #,'BrandName'

In [ ]:
df_Final

In [385]:
df_Final=df_Final.drop_duplicates()
df_Final.reset_index(inplace=True,drop=True)
df_Final.shape

(2635390, 8)

In [386]:
df_Final_all=df_Final 

In [387]:
df_Final_all['Key']=df_Final_all["PartNumber"] + " " + df_Final_all["PartName"] + " " + df_Final_all["Attributes"]

In [ ]:
df_Final_all

In [389]:
df_Final_all_filtered = df_Final_all.sort_values(by='Value', na_position='last').drop_duplicates(subset=['Key'], keep='first')

In [390]:
df_Final_all_filtered.shape

(2592360, 9)

In [ ]:
df_Final_all_filtered

In [391]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [df_Final_all_filtered.iloc[i:i + chunk_size] for i in range(0, len(df_Final_all_filtered), chunk_size)]

In [392]:
df_Final=df_Final.groupby(['Brand','PartNumber','PartName'], as_index=False).agg(
    Total_Autocare_Attributes=('AC_AttributeCount','sum'),
    Autocare_Attributes_Currently_Available=('Attribute_Value','sum'),
    Total_Available_Attributes=('Attribute_Type','count')
)

In [393]:
df_Final['Filled%']=df_Final['Autocare_Attributes_Currently_Available']/df_Final['Total_Autocare_Attributes']
df_Final['FBG_Attributes']=df_Final['Total_Available_Attributes']-df_Final['Autocare_Attributes_Currently_Available']

In [ ]:
df_Final

## Cleaning and Exporting.

In [ ]:
Source="PDM"
Brand="Repair"
Outputfile=fr'Import files\{Source}\{Brand}_SKU_AttrPerc.xlsx'

In [397]:
with pd.ExcelWriter(Outputfile) as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)
    df_Final.to_excel(writer,index=False, sheet_name='Fillinginfo')